# Module 10 · Solutions

In [ ]:
import pandas as pd, numpy as np
rng = np.random.default_rng(42)
BASE = "data/"
px = pd.read_csv(BASE+"nifty50_prices.csv", parse_dates=["date"]).set_index("date").sort_index()
rets_s = px["close"].pct_change().dropna()
rets = rets_s.values

## 10A

In [ ]:
# Ex1 - seed experiment
for seed in [42, 7]:
    r = np.random.default_rng(seed)
    mu, sigma = rets.mean(), rets.std()
    S0 = px["close"].iloc[-1]
    paths = S0*np.cumprod(1 + mu + sigma*r.standard_normal((10_000, 252)), axis=1)
    f = paths[:,-1]
    print(f"seed {seed}: median {np.median(f):,.0f} | 5th pct {np.quantile(f,.05):,.0f} | P(down) {(f<S0).mean():.1%}")
print("Numbers shift ~0.5-1%; every CONCLUSION stands. Policy: report percentiles with the seed and N")
print("disclosed, and never quote MC results to more precision than the seed-to-seed wobble supports.")

# Ex2 - two regimes
calm = rets_s[:"2022-12-31"]; choppy = rets_s["2023-01-01":]
S0 = px["close"].iloc[-1]
for name, r_ in [("calm-era params", calm), ("choppy-era params", choppy)]:
    p = S0*np.cumprod(1 + r_.mean() + r_.std()*rng.standard_normal((10_000,252)), axis=1)
    print(f"{name}: 5th pct {np.quantile(p[:,-1],.05):,.0f}")
print("Materially different downsides from the same history, split by regime. For a risk committee: show the")
print("choppier cone (conservatism) WITH the sentence 'assumes the recent regime persists' - never silently blend.")

# Ex3 - additive world
sim = rets.mean() + rets.std()*rng.standard_normal((2_000, 2520))
add_paths = S0 + S0*np.cumsum(sim, axis=1)
print(f"\nAdditive 10-yr paths going NEGATIVE: {(add_paths.min(axis=1)<0).mean():.1%} of paths")
print("Prices below zero - impossible for an index. Compounding (multiplicative) shrinks losses in rupee")
print("terms as the level falls; a -2% day off 5,000 loses less than off 15,000. Structure matters.")

## 10B

In [ ]:
# Ex1 - the honest SIP
MU_R, SIGMA = 0.07, 0.17; mu_m, sig_m = MU_R/12, SIGMA/np.sqrt(12)
def p_lasts(sip, N=4_000, seed=1):
    r = np.random.default_rng(seed)
    R = mu_m + sig_m*r.standard_normal((N, 65*12))
    c = np.zeros(N); alive = np.ones(N, bool)
    for m in range(65*12):
        if m < 35*12: c = c*(1+R[:,m]) + sip
        else:
            c = c*(1+R[:,m]) - 150_000
            alive &= (c > 0); c = np.maximum(c, 0)
    return alive.mean()
for sip in [20_000, 26_000, 30_000, 34_000]:
    print(f"SIP {sip:,}: P(lasts to 90) = {p_lasts(sip):.0%}")
print("\n~Rs 30-34k/month buys 90% confidence vs the Rs 20k the deterministic calculator blessed.")
print("Critique in one line: the calculator prices the AVERAGE lifetime; you must fund the UNLUCKY one.")

In [ ]:
# Ex2 & 3 - freed WACC and correlated inputs
N = 10_000; INVEST, rev0 = 120, 55.0; years = np.arange(1,8)
g = rng.normal(0.10, 0.04, N); m = rng.normal(0.085, 0.015, N)
def npv_calc(g, m, disc):
    revs = rev0*np.cumprod(1+np.tile(g,(7,1)).T, axis=1)
    return (revs*m[:,None]/(1+disc)**years).sum(axis=1) - INVEST
base = npv_calc(g, m, 0.115)
d = rng.normal(0.115, 0.01, N)
free = (rev0*np.cumprod(1+np.tile(g,(7,1)).T,axis=1)*m[:,None]/(1+d[:,None])**years).sum(axis=1) - INVEST
print(f"P(NPV<0): fixed WACC {(base<0).mean():.0%} vs uncertain WACC {(free<0).mean():.0%}")
for name, x in [("growth", g), ("margin", m), ("wacc", d)]:
    print(f"  corr({name}, NPV) = {np.corrcoef(x, free)[0,1]:+.2f}")
print("Growth dominates the simulation tornado; WACC uncertainty barely moves the tail here.")

# correlated growth & margin
z = (g - g.mean())/g.std()
m_corr = 0.085 + 0.015*(0.6*z + np.sqrt(1-0.36)*rng.standard_normal(N))
corr_npv = npv_calc(g, m_corr, 0.115)
print(f"\nP(NPV<0): independent {(base<0).mean():.0%} vs correlated(0.6) {(corr_npv<0).mean():.0%}")
print("Correlation fattens BOTH tails: bad growth now brings bad margin along, and independence -")
print("the default assumption in every quick model - systematically flatters the downside.")

## 10C

In [ ]:
# Ex1 - regime-split VaR
PORT = 10e7; N = 100_000
for name, r_ in [("calm 2021-22", rets_s[:"2022-12-31"].values), ("choppy 2023+", rets_s["2023-01-01":].values)]:
    v = np.quantile(-rng.choice(r_, N)*PORT, .99)
    print(f"{name}: 99% VaR Rs {v/1e5:.1f} L")
print("Report the choppy-window figure with: 'calibrated to the current volatility regime'.")

# Ex2 - block bootstrap
starts = rng.integers(0, len(rets)-10, N)
blocks = np.stack([rets[s:s+10] for s in starts])
ret10_block = np.prod(1+blocks, axis=1) - 1
iid = np.take(rets, rng.integers(0, len(rets), (N,10)))
ret10_iid = np.prod(1+iid, axis=1) - 1
print(f"\n10-day 99% VaR: iid Rs {np.quantile(-ret10_iid*PORT,.99)/1e5:.1f} L vs block Rs {np.quantile(-ret10_block*PORT,.99)/1e5:.1f} L")
print("Blocks keep 9B's volatility clustering - bad days arrive together - so the block tail is wider. Honest costs more.")

In [ ]:
# Ex3 - backtest the VaR
r = rets_s
breaches, dates = [], []
for i in range(250, len(r)-1):
    var95 = np.quantile(-r.iloc[i-250:i], .95)
    breaches.append(-r.iloc[i] > var95)
breaches = pd.Series(breaches, index=r.index[251:])
print(f"Breach rate: {breaches.mean():.1%} (target ~5%)")
by_year = breaches.groupby(breaches.index.year).mean()
print((by_year*100).round(1).to_string())
print("\nCalibrated overall - but breaches CLUSTER in the regime-shift years: trailing-window VaR is always")
print("late to a new regime. That lag is VaR's known weakness, and the reason desks pair it with stress tests.")